# Notebook 4: Working with API Data
**Time: 60-90 minutes**

In Notebook 3 you made your first `requests.get()` calls and got a brief look at `params=` and `headers=` in §3.6. This notebook expands on those: filtering with query parameters in depth, talking to a second API, and handling the realities of working APIs (auth, paging, errors).

### What you'll learn
- Use query parameters to filter API responses
- Read API documentation effectively
- Understand API authentication (API keys)
- Handle paginated responses
- Work with multiple APIs

In [ ]:
import requests

---
## 4.1 Query parameters

In Notebook 3 §3.6 you saw this line once:

```python
params = {"fields": "name,capital"}
country_response = requests.get(base_url, params=params, headers=headers)
```

Query parameters are how you tell an API *what slice* of data you want — only certain fields, only countries in Europe, only results from page 2. Without them you'd download everything and filter in Python afterward, which is slow and wastes bandwidth.

That `params` dict is a **query parameter** — extra `key=value` pairs the API uses to filter or shape the response. In a URL they appear after a `?`, like `?fields=name,capital`. The `requests` library will encode them for you when you pass `params=`.

Here are the two ways to send them. The first builds the URL by hand; the second uses `params=`:

In [ ]:
# Way 1: Everything in the URL (works but gets messy)
response = requests.get("https://restcountries.com/v3.1/all?fields=name,capital")
print(f"Way 1 - Status: {response.status_code}, Items: {len(response.json())}")

# Way 2: Using params (cleaner, especially with multiple parameters)
response = requests.get(
    "https://restcountries.com/v3.1/all",
    params={"fields": "name,capital"}
)
print(f"Way 2 - Status: {response.status_code}, Items: {len(response.json())}")

Both do the same thing, but the second approach is cleaner. Use `params=` whenever you have query parameters — `requests` handles the encoding for you.

**Exercise.** Fetch all countries, but only the `name` and `region` fields. Then grab the first item from the response.

Required variables:
- `countries_basic` — the full list returned by `.json()`
- `first_country` — the first item in `countries_basic` (a dict)

End the cell by printing both so you can see what you built.

In [ ]:
# YOUR CODE HERE
# 1) Call the REST Countries 'all' endpoint with params={"fields": "name,region"}
# 2) Save the parsed JSON to countries_basic
# 3) Grab the first item into first_country

countries_basic = None
first_country = None

print(countries_basic)
print(first_country)

---
## 4.2 Combining f-string paths with `params=`

Query parameters handle the `?key=value` bits at the end of a URL. But sometimes the **endpoint itself** depends on a value — the region name, a currency, a country. Where do those go?

Rule of thumb:
- **Path segments** (the bit before the `?`) → use an f-string.
- **Query bits** (after the `?`) → use `params=`.

Don't try to glue both into one string by hand — `requests` URL-encodes values for you when you use `params=`, so things like spaces and ampersands never break the URL.

In [ ]:
# Pattern: f-string for the path part, params= for the query bits.
region = "europe"

response = requests.get(
    f"https://restcountries.com/v3.1/region/{region}",
    params={"fields": "name,capital"},
)
print(f"Status: {response.status_code}, Items: {len(response.json())}")

**Exercise.** Use the same split (f-string path + `params=`) to fetch all countries in **Western Europe** with only the `name` and `capital` fields.

Endpoint: `https://restcountries.com/v3.1/subregion/{subregion}`

Required variable:
- `western_europe_countries` — the list returned by `.json()`

Print the length of the list and the first item so you can sanity-check the result.

In [ ]:
# YOUR CODE HERE
# 1) Use the subregion endpoint with f-string for the {subregion} part
# 2) Pass fields=name,capital via params=
# 3) Save the parsed JSON to western_europe_countries

subregion = "western europe"

western_europe_countries = None

---
## 4.3 A second API: Open-Meteo weather

Let's try a different API. Open-Meteo provides weather data — free, no sign-up required, and built around query parameters. Notice the call below takes **three** params (`latitude`, `longitude`, `current_weather`). Hand-gluing three values — each properly URL-encoded and joined with `&` — gets old fast. Passing a dict to `params=` keeps it readable as you add options.

In [ ]:
# Get the current weather for Amsterdam
# (using the coordinates from our detective clues!)
response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
        "latitude": 52.37,
        "longitude": 4.89,
        "current_weather": True
    }
)

weather = response.json()
temp = weather["current_weather"]["temperature"]
windspeed = weather["current_weather"]["windspeed"]
print(f"Amsterdam right now: {temp}°C, wind {windspeed} km/h")

Notice the pattern is the same as REST Countries:
1. `requests.get()` with a URL and parameters
2. `.json()` to get the data
3. Navigate the nested structure to find what you need

**Exercise.** Get the current weather for Brussels (lat 50.85, lon 4.35) and Berlin (lat 52.52, lon 13.40).

Required variables:
- `brussels_temp` — current temperature for Brussels (a number)
- `berlin_temp` — current temperature for Berlin (a number)

Print both so you can see the values.

In [ ]:
# YOUR CODE HERE
# 1) Call Open-Meteo for Brussels (lat=50.85, lon=4.35) with current_weather=True
# 2) Drill into ["current_weather"]["temperature"] and save as brussels_temp
# 3) Do the same for Berlin (lat=52.52, lon=13.40) -> berlin_temp

brussels_temp = None
berlin_temp = None

print(f"Brussels: {brussels_temp}°C")
print(f"Berlin: {berlin_temp}°C")

---
## 4.4 Reading API documentation

When using a new API, look for:

1. **Base URL** — The starting point for all endpoints
2. **Available endpoints** — What data can you request?
3. **Parameters** — How can you filter or customize?
4. **Authentication** — Do you need an API key?
5. **Response format** — What does the data look like?

The REST Countries API documents its endpoints at: https://restcountries.com/

Some endpoints we haven't tried yet:
- `/v3.1/region/{region}` — Countries in a region (e.g., `europe`, `asia`)
- `/v3.1/subregion/{subregion}` — Countries in a subregion (e.g., `western europe`)
- `/v3.1/lang/{language}` — Countries that speak a language
- `/v3.1/currency/{currency}` — Countries using a currency

In [ ]:
# Demo: which countries speak Spanish? Use the /lang/{language} endpoint.
response = requests.get(
    "https://restcountries.com/v3.1/lang/spanish",
    params={"fields": "name,capital"},
)
spanish_speaking = response.json()

print(f"Status: {response.status_code}, found {len(spanish_speaking)} Spanish-speaking countries")
for country in spanish_speaking[:5]:
    name = country["name"]["common"]
    capital = country["capital"][0] if country.get("capital") else "—"
    print(f"  {name} — {capital}")
print(f"  ... and {max(len(spanish_speaking) - 5, 0)} more")

**Exercise.** Look at the endpoint list above and pick the one that lets you find all countries that use a particular currency. Use it to find every country that uses the **Euro** (`eur`).

Hint: the right endpoint takes one value in the path. You can still pass `fields=name,capital` via `params=` to keep the response small.

Required variable:
- `euro_countries` — the list returned by `.json()`

Print the count and the first few entries.

In [ ]:
# YOUR CODE HERE
# 1) Pick the right endpoint from the docs list above
# 2) Plug the currency value into the path
# 3) (Optional) pass fields=name,capital via params= to keep the response small
# 4) Save the parsed JSON to euro_countries

euro_countries = None

print(f"Found {len(euro_countries) if isinstance(euro_countries, list) else 0} countries")

---
## 4.5 Authentication with API keys

The APIs we've used so far are completely open. But many APIs (especially at work) require authentication to:
- Track who's using the API
- Prevent abuse
- Control access to data

The most common way is an **API key** — a secret string that identifies you. Let's try a real one: NASA's Astronomy Picture of the Day. NASA accepts a special `DEMO_KEY` value out of the box for casual experiments, but you can also sign up at https://api.nasa.gov/ for a personal key in under a minute (no credit card).

In [ ]:
# DEMO_KEY works without registering, but rate-limits hard (30/hr, 50/day).
# For real use, get your own key at https://api.nasa.gov/

try:
    api_key = input("Enter your NASA API key (or press Enter for DEMO_KEY): ").strip() or "DEMO_KEY"
except Exception:
    api_key = "DEMO_KEY"  # input() unavailable (non-interactive run)

response = requests.get(
    "https://api.nasa.gov/planetary/apod",
    params={"api_key": api_key},
)
apod = response.json()
print(f"Status: {response.status_code} | Key: {'DEMO_KEY' if api_key == 'DEMO_KEY' else 'personal'}")
print(f"Title: {apod['title']}")
print(f"URL:   {apod['url']}")

**Never put real API keys directly in code you share.** That's why the cell above uses `input()` — you type the key once, it lives in memory for the session, but it never gets saved into the notebook file.

Two patterns for sending the key:
- **As a query parameter** (what NASA does): `params={"api_key": api_key}`
- **In a header** (more common at work): `headers={"Authorization": f"Bearer {api_key}"}`

```python
# Method 1 — query param (used above)
requests.get(url, params={"api_key": api_key})

# Method 2 — header
requests.get(url, headers={"Authorization": f"Bearer {api_key}"})
```

Most workplace APIs give you a personal key and prefer the header form, since URLs with keys can end up in server logs.

---
## 4.6 Fetching multiple items with loops

APIs usually return one "thing" per call — one country, one weather forecast. When you need data for a list of items, the simplest pattern is a loop that calls the API once per item. It's verbose, but it's also explicit: easy to read, easy to debug, and easy to add skip/retry logic later.

In [ ]:
# Get data for several countries
country_names = ["netherlands", "belgium", "germany", "france"]

results = []
for name in country_names:
    # fullText=true forces an exact name match — otherwise "netherlands" also
    # matches "Caribbean Netherlands" and the first hit may surprise you.
    response = requests.get(
        f"https://restcountries.com/v3.1/name/{name}",
        params={"fullText": "true"},
    )
    if response.status_code == 200:  # 200 = OK (see §3.5)
        country = response.json()[0]
        results.append({
            "name": country["name"]["common"],
            "population": country["population"]
        })
        print(f"Got data for {country['name']['common']}")
    else:
        print(f"Failed to get {name}: {response.status_code}")

print(f"\nCollected data for {len(results)} countries")

**Exercise.** Adapt the loop above to fetch a different group of countries and produce a **dict** mapping each country's common name to its population.

Use this list:

```python
country_names = ["spain", "italy", "portugal", "greece"]
```

Required variable:
- `country_populations` — a dict like `{"Spain": 47000000, "Italy": ...}`

Print it.

In [ ]:
# YOUR CODE HERE
# Loop over country_names; for each one, fetch and add to country_populations.

country_names = ["spain", "italy", "portugal", "greece"]
country_populations = {}

print(country_populations)

---
## 4.7 Pagination

Many APIs don't return everything at once. They split results into **pages**. Returning thousands of items in one response would be slow, large, and easy to time out — paging keeps each response small and predictable.

Two common parameter styles:

```python
# Offset-based: "start at item 0, give me 100 items"
params = {"offset": 0, "limit": 100}

# Page-based: "give me page 1, with 100 items per page"
params = {"page": 1, "per_page": 100}
```

Below we'll use the **PokéAPI**, which is free, public, and paginates with `limit`/`offset`. Its responses also tell you the total count and the URL of the next page.

In [ ]:
# PokéAPI paginates with limit + offset.
response = requests.get(
    "https://pokeapi.co/api/v2/pokemon",
    params={"limit": 5, "offset": 0},
)
data = response.json()

print(f"Total Pokémon in database: {data['count']}")
print(f"This page returned: {len(data['results'])} results")
print(f"Next page URL: {data['next']}")
print()
print("Names on this page:")
for p in data["results"]:
    print(f"  {p['name']}")

**Exercise.** Use PokéAPI to fetch the **first 30 Pokémon names** in two pages of 15. Make two calls — one with `offset=0` and one with `offset=15`, both with `limit=15` — then combine the names into one list.

Required variable:
- `pokemon_names` — a list of 30 lowercase strings (the `name` field from each result)

Print the length and the first three names.

In [ ]:
# YOUR CODE HERE
# 1) Call PokéAPI with limit=15 and offset=0  → first page
# 2) Call PokéAPI with limit=15 and offset=15 → second page
# 3) From each response, take the "name" of every item in results
# 4) Combine into pokemon_names (should have 30 entries)

pokemon_names = []

print(f"Got {len(pokemon_names)} names")
print(f"First three: {pokemon_names[:3]}")

---
## 4.8 Checkpoint

Run this cell to verify your understanding. It checks every variable you produced across the exercises.

In [ ]:
# Run this cell to check your understanding.

required_vars = [
    "countries_basic",
    "first_country",
    "western_europe_countries",
    "brussels_temp",
    "berlin_temp",
    "euro_countries",
    "country_populations",
    "pokemon_names",
]

missing = [v for v in required_vars if v not in globals() or globals()[v] is None]
assert not missing, f"Missing or unfinished: {missing}. Go back and complete those exercises."

# 4.1 - countries_basic + first_country
assert isinstance(countries_basic, list) and len(countries_basic) > 100, (
    f"countries_basic should be a list of 100+ countries, got "
    f"{type(countries_basic).__name__} len="
    f"{len(countries_basic) if isinstance(countries_basic, list) else 'n/a'}"
)
assert isinstance(first_country, dict) and "name" in first_country, (
    "first_country should be a dict with a 'name' key"
)

# 4.2 - western_europe_countries (Western Europe has ~9 entries)
assert isinstance(western_europe_countries, list) and 5 <= len(western_europe_countries) <= 15, (
    f"western_europe_countries should have ~7-10 entries, got "
    f"{len(western_europe_countries) if isinstance(western_europe_countries, list) else 'n/a'}"
)

# 4.3 - Brussels + Berlin temps
assert isinstance(brussels_temp, (int, float)), f"brussels_temp should be a number, got {type(brussels_temp).__name__}"
assert isinstance(berlin_temp, (int, float)), f"berlin_temp should be a number, got {type(berlin_temp).__name__}"
assert -30 <= brussels_temp <= 45, f"brussels_temp looks unrealistic: {brussels_temp}"
assert -30 <= berlin_temp <= 45, f"berlin_temp looks unrealistic: {berlin_temp}"

# 4.4 - euro_countries
assert isinstance(euro_countries, list) and len(euro_countries) >= 15, (
    f"euro_countries should be a list of 15+ countries, got "
    f"{len(euro_countries) if isinstance(euro_countries, list) else 'n/a'}"
)

# 4.6 - country_populations
assert isinstance(country_populations, dict) and len(country_populations) == 4, (
    f"country_populations should have all 4 countries, got "
    f"{len(country_populations) if isinstance(country_populations, dict) else 'n/a'}: {country_populations}"
)
assert all(isinstance(v, int) and v > 0 for v in country_populations.values()), (
    "All population values should be positive integers"
)

# 4.7 - pokemon_names
assert isinstance(pokemon_names, list) and len(pokemon_names) == 30, (
    f"pokemon_names should have exactly 30 entries, got "
    f"{len(pokemon_names) if isinstance(pokemon_names, list) else 'n/a'}"
)
assert all(isinstance(n, str) for n in pokemon_names), (
    "All entries in pokemon_names should be strings (the 'name' field of each result)"
)

notebook4_ready_for_clue = True
print("All checks passed — you're ready for the clue.")

---
## 4.9 Data Detective Challenge — Clue 4

What's the **total population of Europe**? (Sum the populations of every country in the region.)

In [ ]:
# CHALLENGE: total population of Europe
# Fetch all European countries with their population, then sum them.

assert notebook4_ready_for_clue, "Run the 4.8 checkpoint first."

response = requests.get(
    "https://restcountries.com/v3.1/region/europe",
    params={"fields": "name,population"},
)
data = response.json()

clue_4 = ___  # Sum the 'population' values from each item in data

print(f"Your answer: {clue_4:,} people in Europe")

# --- Verification ---
if 700_000_000 <= clue_4 <= 800_000_000:
    print(f"\n>>> Clue 4 collected!")
    print(f"Write down: clue_4 = {clue_4}")
else:
    print("Not quite — expected ~745 million.")
    print("Hint: each item in `data` has a 'population' key. sum() them.")

---
**Next up:** [Notebook 5 — Transforming and Saving Data](5_transforming_and_saving_data.ipynb)